# Edge Inspection: Cross-Backend Comparison

Side-by-side comparison of weighted directed graph edges across GDF, SQL, and PostGIS backends.

#### Data Sources

- **Per-feature CSVs**: Pre-computed by `compare_graphs.py --per-feature` (one CSV per numeric column)
- **Weighted graph GeoPackages**: Original source files loaded directly via fiona

#### Workflow

1. **Configuration** — Set paths and edge IDs to inspect
2. **Load Data** — Load CSVs and/or GeoPackages
3. **Summary** — Failure counts per feature (from CSVs)
4. **Inspect Edge (CSV)** — Per-feature comparison for specific edges
5. **Inspect Edge (Graph)** — Raw attribute comparison from GeoPackages
6. **Cross-Check** — Verify CSV values match raw GeoPackage data
7. **Filter Failing Edges** — List edges exceeding tolerance per feature

## 1. Configuration

Set data source paths, edge IDs to inspect, and display options.
Enable either or both data sources depending on what you need.

In [ ]:
# =============================================================================
# CONFIGURATION - Adjust these settings before running
# =============================================================================

# --- Per-Feature CSV Source ---
# Output folder from: compare_graphs.py --per-feature
LOAD_CSV: bool = True
FEATURE_DIR: str = "../../output/feature_comparison_20260430_151924"

# --- Weighted Graph Source (GeoPackage files) ---
# Original directed graph GeoPackages for independent verification
LOAD_GRAPHS: bool = True
GDF_GPKG: str =         "../../data/compare_graphs/test_graph_directed_mem_v23w.gpkg"
SQL_GPKG: str =         "../../data/compare_graphs/test_graph_directed_sql_v24w2.gpkg"
POSTGIS_GPKG: str = "../../data/compare_graphs/test_graph_directed_postgis_v24w2.gpkg"

# --- Edge IDs to Inspect ---
# Specific edges to examine in detail
EDGE_IDS: list[int] = [1375,1378,1385,43949]

# --- Display ---
MAX_FAIL_ROWS: int = 20  # Max failing edges to show per feature

# Columns to exclude from graph comparison (topology/internal)
SKIP_COLS = {"_idx_id", "geometry", "fid"}
TOPOLOGY_COLS = {
    "source_id", "target_id", "source_str", "target_str",
    "source_x", "source_y", "target_x", "target_y",
}

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print("=" * 70)
print("Configuration Loaded")
print("=" * 70)
print(f"Load CSVs:   {LOAD_CSV}" + (f"  ({FEATURE_DIR})" if LOAD_CSV else ""))
print(f"Load Graphs: {LOAD_GRAPHS}")
if LOAD_GRAPHS:
    print(f"  GDF:     {GDF_GPKG}")
    print(f"  SQL:     {SQL_GPKG}")
    print(f"  PostGIS: {POSTGIS_GPKG}")
print(f"Edge IDs:    {EDGE_IDS}")
print("=" * 70)

### 1.1 Imports

In [ ]:
from pathlib import Path
import sys
import os

import fiona
import pandas as pd

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)


print("=" * 70)
print("All imports loaded successfully")
print("=" * 70)

## 2. Load Data

Load per-feature CSVs and/or weighted graph GeoPackages based on configuration.

In [ ]:
# --- Load Per-Feature CSVs ---
feature_data: dict[str, pd.DataFrame] = {}
summary_rows = []

CSV_DTYPES = {
    "Edge_ID": int,
    "GDF": float,
    "SQL": float,
    "PostGIS": float,
    "Delta": float,
    "Delta_pct": float,
    "Tolerance_Check": str,
}

if LOAD_CSV:
    feature_dir = Path(FEATURE_DIR)
    if not feature_dir.exists():
        raise FileNotFoundError(f"Feature directory not found: {feature_dir}")

    csv_files = sorted(feature_dir.glob("*.csv"))
    print(f"Loading {len(csv_files)} per-feature CSVs from {feature_dir.name}/")

    for csv_path in csv_files:
        feature_name = csv_path.stem
        df = pd.read_csv(csv_path, dtype=CSV_DTYPES)
        feature_data[feature_name] = df

        n_total = len(df)
        n_fail = (df["Tolerance_Check"] == "FAIL").sum()
        n_na = (df["Tolerance_Check"] == "N/A").sum()
        max_delta_pct = df["Delta_pct"].max()

        summary_rows.append({
            "Feature": feature_name,
            "Total": n_total,
            "FAIL": n_fail,
            "FAIL_pct": round(n_fail / n_total * 100, 2) if n_total > 0 else 0,
            "N/A": n_na,
            "Max_Delta_pct": round(max_delta_pct, 6) if pd.notna(max_delta_pct) else 0,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("FAIL_pct", ascending=False).reset_index(drop=True)
    print(f"  Loaded {len(feature_data)} features, {summary_rows[0]['Total']:,} edges each")
else:
    summary_df = pd.DataFrame()
    print("CSV loading disabled (LOAD_CSV=False)")

In [ ]:
# --- Load Weighted Graph GeoPackages ---
graphs: dict[str, pd.DataFrame] = {}

if LOAD_GRAPHS:
    graph_sources = {"GDF": GDF_GPKG, "SQL": SQL_GPKG, "PostGIS": POSTGIS_GPKG}

    for label, gpkg_path in graph_sources.items():
        path = Path(gpkg_path)
        if not path.exists():
            print(f"  WARNING: {label} GeoPackage not found: {path}")
            continue

        print(f"  Loading {label}...", end=" ", flush=True)
        records = []
        with fiona.open(str(path), layer="edges") as src:
            for feat in src:
                records.append(dict(feat["properties"]))
        df = pd.DataFrame(records)
        if "id" in df.columns:
            df = df.set_index("id")
        graphs[label] = df
        print(f"{len(df):,} edges, {len(df.columns)} cols")

    print(f"\n  Loaded {len(graphs)} graph backends")
else:
    print("Graph loading disabled (LOAD_GRAPHS=False)")

## 3. Summary — Failures per Feature (CSV)

Overview of which features diverge most across backends. Sorted by failure percentage.

In [ ]:
if not summary_df.empty:
    display(summary_df.style.format({
        "FAIL_pct": "{:.2f}%",
        "Max_Delta_pct": "{:.6f}",
        "Total": "{:,}",
        "FAIL": "{:,}",
        "N/A": "{:,}",
    }).bar(subset=["FAIL_pct"], color="#ff6b6b", vmin=0, vmax=100))
else:
    print("No CSV data loaded — enable LOAD_CSV to see summary")

## 4. Inspect Edge — CSV View

For each edge ID, show all features side-by-side from the per-feature CSVs.
Each row = one feature column, columns = backend values + delta + tolerance.

In [ ]:
def inspect_edge_csv(edge_id: int) -> pd.DataFrame:
    """Build a per-feature comparison table for a single edge from CSV data."""
    rows = []
    for feature_name, df in sorted(feature_data.items()):
        edge_row = df[df["Edge_ID"] == edge_id]
        if edge_row.empty:
            continue
        r = edge_row.iloc[0]
        rows.append({
            "Feature": feature_name,
            "GDF": r.get("GDF"),
            "SQL": r.get("SQL"),
            "PostGIS": r.get("PostGIS"),
            "Delta": r.get("Delta"),
            "Delta_pct": r.get("Delta_pct"),
            "Tolerance": r.get("Tolerance_Check"),
        })
    return pd.DataFrame(rows).set_index("Feature")


if feature_data:
    for eid in EDGE_IDS:
        print(f"\n{'=' * 70}")
        print(f"  Edge ID: {eid} (CSV)")
        print(f"{'=' * 70}")
        result = inspect_edge_csv(eid)
        if result.empty:
            print(f"  Edge {eid} not found in CSV data")
        else:
            display(result)
else:
    print("No CSV data loaded")

## 5. Inspect Edge — Graph View

Direct comparison from weighted graph GeoPackages. Independent of `compare_graphs.py`.
Each row = one attribute column, columns = backend values.

In [ ]:
def inspect_edge_graph(edge_id: int) -> pd.DataFrame:
    """Build a side-by-side attribute table for a single edge from graph GeoPackages."""
    data = {}
    for label, df in graphs.items():
        if edge_id in df.index:
            data[label] = df.loc[edge_id]

    if not data:
        return pd.DataFrame()

    result = pd.DataFrame(data)
    # Exclude internal/topology columns
    exclude = SKIP_COLS | TOPOLOGY_COLS
    result = result[~result.index.isin(exclude)]
    return result


if graphs:
    for eid in EDGE_IDS:
        print(f"\n{'=' * 70}")
        print(f"  Edge ID: {eid} (Graph)")
        print(f"{'=' * 70}")
        result = inspect_edge_graph(eid)
        if result.empty:
            print(f"  Edge {eid} not found in graph data")
        else:
            display(result)
else:
    print("No graph data loaded")

## 6. Cross-Check: CSV vs Graph

Verify that per-feature CSV values match raw GeoPackage data for the inspected edges.
Any mismatch indicates a bug in `compare_graphs.py` pipeline.

In [ ]:
if feature_data and graphs:
    mismatches = []

    for eid in EDGE_IDS:
        for feature_name, csv_df in feature_data.items():
            csv_row = csv_df[csv_df["Edge_ID"] == eid]
            if csv_row.empty:
                continue
            csv_row = csv_row.iloc[0]

            for label in graphs:
                if label not in csv_row.index:
                    continue
                if eid not in graphs[label].index:
                    continue
                if feature_name not in graphs[label].columns:
                    continue

                csv_val = csv_row[label]
                graph_val = graphs[label].loc[eid, feature_name]

                # Convert to numeric for comparison
                csv_num = pd.to_numeric(csv_val, errors="coerce")
                graph_num = pd.to_numeric(graph_val, errors="coerce")

                if pd.isna(csv_num) and pd.isna(graph_num):
                    continue
                if pd.isna(csv_num) != pd.isna(graph_num):
                    mismatches.append({
                        "Edge_ID": eid, "Feature": feature_name, "Backend": label,
                        "CSV": csv_val, "Graph": graph_val, "Issue": "NaN mismatch",
                    })
                elif abs(csv_num - graph_num) > 1e-10:
                    mismatches.append({
                        "Edge_ID": eid, "Feature": feature_name, "Backend": label,
                        "CSV": csv_val, "Graph": graph_val, "Issue": "Value mismatch",
                    })

    if mismatches:
        print(f"MISMATCHES FOUND: {len(mismatches)}")
        display(pd.DataFrame(mismatches))
    else:
        print(f"All values match between CSV and Graph for edges {EDGE_IDS}")
else:
    print("Both CSV and Graph data required for cross-check")

## 7. Filter Failing Edges

For features with failures, show the first N failing edges with their backend values.

In [ ]:
if feature_data:
    # Only show features that have failures
    failing_features = summary_df[summary_df["FAIL"] > 0]["Feature"].tolist()

    if not failing_features:
        print("No failing features found")
    else:
        print(f"Features with failures: {len(failing_features)}")

        for feature_name in failing_features:
            df = feature_data[feature_name]
            fails = df[df["Tolerance_Check"] == "FAIL"].head(MAX_FAIL_ROWS)
            n_total_fail = (df["Tolerance_Check"] == "FAIL").sum()

            print(f"\n{'=' * 70}")
            print(f"  {feature_name}: {n_total_fail:,} failing edges"
                  f" (showing first {min(MAX_FAIL_ROWS, len(fails))})")
            print(f"{'=' * 70}")
            display(fails)
else:
    print("No CSV data loaded")